# Chapter 4: Training a Neural Network and Computational Thinking

In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
# At the top of your notebook, add:
np.set_printoptions(suppress=True, precision=8)
torch.set_printoptions(sci_mode=False, precision=8)

cuda


# 1. Understanding Modules and Layers in PyTorch

## Parameter Registration

Correct way to do parameter registration is commented in the following code-snippet and it will run with error.

In [2]:
class BrokenLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # Just a regular tensor
        self.weight = torch.randn(out_dim, in_dim)
        self.bias = torch.zeros(out_dim)
        
        # Correct way.
        # self.weight = nn.Parameter(torch.randn(out_dim, in_dim))
        # self.bias = nn.Parameter(torch.zeros(out_dim))
        
    def forward(self, x):
        return x @ self.weight.T + self.bias

layer = BrokenLayer(10, 5)
print(f"Number of parameters: {sum(p.numel() for p in layer.parameters())}")  # 0
print(f"Requires grad on weight? {layer.weight.requires_grad}")  # False

# The optimizer will have nothing to optimize!
optimizer = torch.optim.SGD(layer.parameters(), lr=0.01)
print(f"Optimizer param groups: {len(optimizer.param_groups[0]['params'])}")  # 0

Number of parameters: 0
Requires grad on weight? False


ValueError: optimizer got an empty parameter list

In [4]:
class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # These are automatically registered as parameters
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.Tensor(out_features))
        
        # Alternative manual registration:
        # self.register_parameter('weight', nn.Parameter(...))

## Module Hierarchy: Parameters

In [5]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        # Child modules are automatically registered
        self.layer1 = nn.Linear(10, 20)
        self.layer2 = nn.Linear(20, 5)
        
    def forward(self, x):
        return self.layer2(self.layer1(x))
        
# Accessing the hierarchy:
model = Network()
print(list(model.children()))  # [layer1, layer2]


[Linear(in_features=10, out_features=20, bias=True), Linear(in_features=20, out_features=5, bias=True)]


In [7]:
print(list(model.parameters()))  # All parameters from both layers

[Parameter containing:
tensor([[ 0.04497739,  0.10519058, -0.21158044, -0.09482440, -0.30024675,
          0.22213356, -0.28273168, -0.02712731,  0.30192059, -0.04642568],
        [ 0.07350341,  0.12941694,  0.10103288,  0.26430675, -0.11819174,
         -0.31233934,  0.07395262,  0.24144739, -0.01277983,  0.14603227],
        [-0.29200214, -0.30849111,  0.30748573,  0.14315876, -0.13794115,
         -0.27202782,  0.20273744,  0.21430697,  0.29269651,  0.27908841],
        [ 0.11304384,  0.25066820, -0.27193803,  0.01297582,  0.21171223,
         -0.27442485,  0.14900565, -0.10631216, -0.01539421,  0.15864326],
        [-0.17846566,  0.23887576,  0.01952391, -0.02924793, -0.15678969,
          0.25516492,  0.20870945,  0.10403392,  0.04524674,  0.16955328],
        [-0.24551478,  0.00438887, -0.11636395, -0.10766737, -0.03042062,
          0.15053473, -0.09813464,  0.29482856,  0.24405615,  0.23473240],
        [ 0.24710485, -0.18842036,  0.23674311,  0.05264667, -0.25172254,
         

In [6]:
class CustomLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_features, out_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
    
    def forward(self, x):
        return x @ self.weight + self.bias

## An Example of a Complex Network

In [8]:
class ComplexNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Submodule 1: Feature extractor
        self.feature_extractor = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Submodule 2: Processor
        self.processor = nn.ModuleList([
            nn.Linear(256, 128),
            nn.Linear(128, 64)
        ])
        
        # Submodule 3: Output head
        self.classifier = nn.Linear(64, 10)
        
    def forward(self, x):
        x = self.feature_extractor(x)
        for layer in self.processor:
            x = layer(x)
        return self.classifier(x)

# Inspecting the hierarchy
model = ComplexNetwork()
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Module structure:\n{model}")

# Accessing specific parts
print(f"Feature extractor parameters: {list(model.feature_extractor.parameters())}")

Number of parameters: 242762
Module structure:
ComplexNetwork(
  (feature_extractor): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (processor): ModuleList(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): Linear(in_features=128, out_features=64, bias=True)
  )
  (classifier): Linear(in_features=64, out_features=10, bias=True)
)
Feature extractor parameters: [Parameter containing:
tensor([[-0.00808910, -0.01115680, -0.00184645,  ...,  0.02986764,
         -0.00351129, -0.00637750],
        [ 0.02537426, -0.00611338, -0.02414018,  ..., -0.03432127,
         -0.01742587, -0.03312283],
        [-0.00606414,  0.01290857, -0.02665715,  ..., -0.00376067,
         -0.01674598, -0.01327974],
        ...,
        [ 0.00357364, -0.01545253, -0.01199080,  ...,  0.01447603,
          0.00363764,  0.00449602],
        [ 0.03177841,  0.00434683,  0.00694681,  ..., -0.02247404,
          0.0

## A Layer with a Conditional Parameter

In [9]:
class ConditionalLayer(nn.Module):
    """Layer that sometimes has bias, sometimes doesn't"""
    def __init__(self, in_dim, out_dim, use_bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_dim, in_dim))
        self.use_bias = use_bias
        
        if use_bias:
            self.bias = nn.Parameter(torch.zeros(out_dim))
        else:
            # Important: Register bias as None
            self.register_parameter('bias', None)
            
    def forward(self, x):
        result = x @ self.weight.T
        if self.bias is not None:
            result = result + self.bias
        return result

# Both versions work correctly with optimizers
layer1 = ConditionalLayer(10, 5, use_bias=True)
layer2 = ConditionalLayer(10, 5, use_bias=False)
print(f"Layer1 params: {len(list(layer1.parameters()))}")  # 2
print(f"Layer2 params: {len(list(layer2.parameters()))}")  # 1

Layer1 params: 2
Layer2 params: 1


## Hooks

### Forward Hook

In [10]:
import torch
import torch.nn as nn

# Define a simple model
model = nn.Linear(5, 3)

# Define a hook function
def print_output(module, input, output):
    print(module)
    print(f"Iput: {input}")
    print(f"Output: {output}")

# Register the hook
handle = model.register_forward_hook(print_output)

# Run the model
x = torch.randn(1, 5)
print(x)
output = model(x)

# Remove the hook
handle.remove()

tensor([[ 0.24239229,  0.83586425, -0.33849525, -0.24874426, -0.02735049]])
Linear(in_features=5, out_features=3, bias=True)
Iput: (tensor([[ 0.24239229,  0.83586425, -0.33849525, -0.24874426, -0.02735049]]),)
Output: tensor([[0.55133754, 0.06624623, 0.41174358]], grad_fn=<AddmmBackward0>)


### Backward Hook

In [11]:
import torch
import torch.nn as nn

model = nn.Linear(5, 3)
model.weight.data = torch.tensor([
    [0.1, 0.2, 0.3, 0.4, 0.5],
    [0.6, 0.7, 0.8, 0.9, 1.0],
    [1.1, 1.2, 1.3, 1.4, 1.5]
])
model.bias.data = torch.tensor([0.1, 0.2, 0.3])

def print_gradients(module, grad_input, grad_output):
    print(f"Grad Input: {grad_input}")
    print(f"Grad Output: {grad_output}")

handle = model.register_backward_hook(print_gradients)

x = torch.tensor([[1., 2., 3., 4., 5.]])
print(x)
output = model(x)
loss = output.sum()
loss.backward()

handle.remove()

tensor([[1., 2., 3., 4., 5.]])
Grad Input: (tensor([1., 1., 1.]), None, tensor([[1., 1., 1.],
        [2., 2., 2.],
        [3., 3., 3.],
        [4., 4., 4.],
        [5., 5., 5.]]))
Grad Output: (tensor([[1., 1., 1.]]),)


/home/rkb0022/CPE487587_SP26/Code/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1866: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Grad Input Tuple: (1): initial gradient input (2) Gradient with respect to bias (not provided by Hook), (3) Initial Gradient with respect to W (4) Out gradient with respect to W.

In [12]:
print("Weight gradient:", model.weight.grad)
print("Bias gradient:", model.bias.grad)

Weight gradient: tensor([[1., 2., 3., 4., 5.],
        [1., 2., 3., 4., 5.],
        [1., 2., 3., 4., 5.]])
Bias gradient: tensor([1., 1., 1.])


In [13]:
import torch
import torch.nn as nn

class HookedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 5)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(5, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# 1. Define the Forward Hook (peeking at activations)
def forward_hook_fn(module, input, output):
    print(f"--- Forward Hook: {module.__class__.__name__} ---")
    print(f"Input Shape: {input[0].shape}")
    print(f"Output Shape: {output.shape}\n")

# 2. Define the Backward Hook (peeking at gradients)
def backward_hook_fn(module, grad_input, grad_output):
    print(f"--- Backward Hook: {module.__class__.__name__} ---")
    # grad_output is the gradient of the loss w.r.t the output of this layer
    print(f"Gradient Output Norm: {grad_output[0].norm().item():.4f}\n")

# --- Execution ---
model = HookedModel()

# Register hooks on the ReLU layer specifically
handle_f = model.relu.register_forward_hook(forward_hook_fn)
handle_b = model.relu.register_full_backward_hook(backward_hook_fn)

# Dummy Input
data = torch.randn(1, 10)
target = torch.randn(1, 2)

# Forward Pass
output = model(data)

# Backward Pass
loss = torch.nn.functional.mse_loss(output, target)
loss.backward()

# Clean up (it's good practice to remove hooks after debugging)
handle_f.remove()
handle_b.remove()

--- Forward Hook: ReLU ---
Input Shape: torch.Size([1, 5])
Output Shape: torch.Size([1, 5])

--- Backward Hook: ReLU ---
Gradient Output Norm: 0.2264



In [14]:
# df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE487587_SP26/refs/heads/master/Data/ResourceAssessmentSummaryData032011.csv",
#                 schema_overrides={
#         "Design Head (feet) ": pl.Utf8,
#         "Design Flow (cfs)": pl.Utf8,
#         "Installed Capacity (kW)": pl.Utf8,
#         "Annual Production (MWh)": pl.Utf8,
#         "Plant Factor": pl.Utf8,
#         "Total Construction Cost (1,000 $)": pl.Utf8,
#         "Annual O&M Cost (1,000 $)": pl.Utf8,
#         "Cost per Installed Capacity ($/kW)": pl.Utf8,
#         "IRR with Green Incentives": pl.Utf8,
#     }
# )

# # Remove quotes, commas, and dollar signs, then convert to float
# df = df.with_columns(
#     pl.col([
#         "Design Head (feet) ",
#         "Design Flow (cfs)",
#         "Installed Capacity (kW)",
#         "Annual Production (MWh)",
#         "Plant Factor",
#         "Total Construction Cost (1,000 $)",
#         "Annual O&M Cost (1,000 $)",
#         "Cost per Installed Capacity ($/kW)",
#         "IRR with Green Incentives",
#     ])
#     .str.replace_all(r'["\$,]', '')  # remove quotes, $, and commas
#     .str.replace_all(r'[<>]', '')     # remove < and >
#     .cast(pl.Float64)
# )

In [15]:
# df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE487587_SP26/refs/heads/master/Data/ResourceAssessmentSummaryData032011.csv",
#                 schema_overrides={
#         "Design Head (feet) ": pl.Utf8,
#         "Design Flow (cfs)": pl.Utf8,
#         "Installed Capacity (kW)": pl.Utf8,
#         "Annual Production (MWh)": pl.Utf8,
#         "Plant Factor": pl.Utf8,
#         "Total Construction Cost (1,000 $)": pl.Utf8,
#         "Annual O&M Cost (1,000 $)": pl.Utf8,
#         "Cost per Installed Capacity ($/kW)": pl.Utf8,
#         "IRR with Green Incentives": pl.Utf8,
#     }
# )

# # Remove quotes, commas, and dollar signs, then convert to float
# df = df.with_columns(
#     pl.col([
#         "Design Head (feet) ",
#         "Design Flow (cfs)",
#         "Installed Capacity (kW)",
#         "Annual Production (MWh)",
#         "Plant Factor",
#         "Total Construction Cost (1,000 $)",
#         "Annual O&M Cost (1,000 $)",
#         "Cost per Installed Capacity ($/kW)",
#         "IRR with Green Incentives",
#     ])
#     .str.replace_all(r'["\$,]', '')  # remove quotes, $, and commas
#     .str.replace_all(r'[<>]', '')     # remove < and >
#     .cast(pl.Float64)
# )

# 2. Neural Network Training

In [16]:
df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE486586_FA25/refs/heads/main/Data/Concrete_Compressive_Strength/Concrete_Data.csv")

df

Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
str,str,str,str,str,str,str,str,str
"""540.0 ""","""0.0 ""","""0.0 ""","""162.0 ""","""2.5 ""","""1040.0 ""","""676.0 ""","""28 ""","""79.99 """
"""540.0 ""","""0.0 ""","""0.0 ""","""162.0 ""","""2.5 ""","""1055.0 ""","""676.0 ""","""28 ""","""61.89 """
"""332.5 ""","""142.5 ""","""0.0 ""","""228.0 ""","""0.0 ""","""932.0 ""","""594.0 ""","""270 ""","""40.27 """
"""332.5 ""","""142.5 ""","""0.0 ""","""228.0 ""","""0.0 ""","""932.0 ""","""594.0 ""","""365 ""","""41.05 """
"""198.6 ""","""132.4 ""","""0.0 ""","""192.0 ""","""0.0 ""","""978.4 ""","""825.5 ""","""360 ""","""44.30 """
…,…,…,…,…,…,…,…,…
"""276.4 ""","""116.0 ""","""90.3 ""","""179.6 ""","""8.9 ""","""870.1 ""","""768.3 ""","""28 ""","""44.28 """
"""322.2 ""","""0.0 ""","""115.6 ""","""196.0 ""","""10.4 ""","""817.9 ""","""813.4 ""","""28 ""","""31.18 """
"""148.5 ""","""139.4 ""","""108.6 ""","""192.7 ""","""6.1 ""","""892.4 ""","""780.0 ""","""28 ""","""23.70 """


In [17]:
# Rename
df.columns = ['Cement', 'BlastFuranceSlag', 'FlyAsh', 'Water', 'Superplasticizer', 'CoarseAggregate', 'FineAggregate', 'Age',  'ConcreteStrength']

In [18]:
df

Cement,BlastFuranceSlag,FlyAsh,Water,Superplasticizer,CoarseAggregate,FineAggregate,Age,ConcreteStrength
str,str,str,str,str,str,str,str,str
"""540.0 ""","""0.0 ""","""0.0 ""","""162.0 ""","""2.5 ""","""1040.0 ""","""676.0 ""","""28 ""","""79.99 """
"""540.0 ""","""0.0 ""","""0.0 ""","""162.0 ""","""2.5 ""","""1055.0 ""","""676.0 ""","""28 ""","""61.89 """
"""332.5 ""","""142.5 ""","""0.0 ""","""228.0 ""","""0.0 ""","""932.0 ""","""594.0 ""","""270 ""","""40.27 """
"""332.5 ""","""142.5 ""","""0.0 ""","""228.0 ""","""0.0 ""","""932.0 ""","""594.0 ""","""365 ""","""41.05 """
"""198.6 ""","""132.4 ""","""0.0 ""","""192.0 ""","""0.0 ""","""978.4 ""","""825.5 ""","""360 ""","""44.30 """
…,…,…,…,…,…,…,…,…
"""276.4 ""","""116.0 ""","""90.3 ""","""179.6 ""","""8.9 ""","""870.1 ""","""768.3 ""","""28 ""","""44.28 """
"""322.2 ""","""0.0 ""","""115.6 ""","""196.0 ""","""10.4 ""","""817.9 ""","""813.4 ""","""28 ""","""31.18 """
"""148.5 ""","""139.4 ""","""108.6 ""","""192.7 ""","""6.1 ""","""892.4 ""","""780.0 ""","""28 ""","""23.70 """


Clearly, Polars didn't do a good job of reading dataset properly, so we need to manually strip out white spaces, and read as float.

In [19]:
# 3. Clean and Cast all columns to Float32
df = df.with_columns(
    pl.all().str.strip_chars().cast(pl.Float32)
)
df

Cement,BlastFuranceSlag,FlyAsh,Water,Superplasticizer,CoarseAggregate,FineAggregate,Age,ConcreteStrength
f32,f32,f32,f32,f32,f32,f32,f32,f32
540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28.0,79.989998
540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28.0,61.889999
332.5,142.5,0.0,228.0,0.0,932.0,594.0,270.0,40.27
332.5,142.5,0.0,228.0,0.0,932.0,594.0,365.0,41.049999
198.600006,132.399994,0.0,192.0,0.0,978.400024,825.5,360.0,44.299999
…,…,…,…,…,…,…,…,…
276.399994,116.0,90.300003,179.600006,8.9,870.099976,768.299988,28.0,44.279999
322.200012,0.0,115.599998,196.0,10.4,817.900024,813.400024,28.0,31.18
148.5,139.399994,108.599998,192.699997,6.1,892.400024,780.0,28.0,23.700001


Casting Strings to Float can be problematic, but we are gonna live with it for the time being

In [20]:
import polars as pl

# Float32 has limited precision
value_f32 = pl.Series([79.99]).cast(pl.Float32)[0]
print(value_f32)  # 79.989998 (precision lost!)

79.98999786376953


We want to estimat annual production based BCR, ConstructionCost and DesignHead

In [21]:
class SimpleNN(nn.Module):
    def __init__(self, in_features):
        super(SimpleNN, self).__init__()
        self.in_features = in_features
        self.fc1 = nn.Linear(self.in_features, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 16)
        self.fc4 = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [22]:
# Convert Polars DataFrame to numpy arrays
X = df.drop(['ConcreteStrength']).to_numpy() 
y = df['ConcreteStrength'].to_numpy()     

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [25]:
y.shape

(1030,)

In [26]:
y_train.reshape(-1,1)

array([[27.68],
       [62.05],
       [23.8 ],
       [33.4 ],
       [ 7.4 ],
       [27.77],
       [18.29],
       [48.59],
       [39.7 ],
       [ 4.57],
       [13.29],
       [36.97],
       [22.53],
       [71.3 ],
       [25.61],
       [76.24],
       [62.94],
       [17.54],
       [41.05],
       [21.86],
       [47.13],
       [16.5 ],
       [22.72],
       [29.72],
       [19.93],
       [ 9.62],
       [39.05],
       [42.13],
       [39.32],
       [34.49],
       [28.1 ],
       [38.6 ],
       [53.77],
       [ 7.32],
       [32.82],
       [43.38],
       [55.16],
       [35.23],
       [66.7 ],
       [30.88],
       [76.8 ],
       [17.96],
       [55.06],
       [64.3 ],
       [33.8 ],
       [45.94],
       [37.26],
       [24.85],
       [40.15],
       [13.54],
       [32.88],
       [17.57],
       [21.54],
       [17.84],
       [23.4 ],
       [55.55],
       [17.6 ],
       [31.42],
       [13.82],
       [65.91],
       [81.75],
       [28.8 ],
       [

In [27]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32, device=device)

In [28]:
model = SimpleNN(in_features = X_train_tensor.shape[1]).to(device)


In [29]:
# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)
# Train the model
epochs = 100000

losses = torch.zeros(epochs, device=device)

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    
    losses[epoch] = loss
    optimizer.step()
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')
        print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.5f} MB')

Epoch 1000/100000, Loss: 1499.584716796875
GPU Memory: 17.84863 MB
Epoch 2000/100000, Loss: 1307.7098388671875
GPU Memory: 17.84863 MB
Epoch 3000/100000, Loss: 979.979736328125
GPU Memory: 17.84863 MB
Epoch 4000/100000, Loss: 614.1906127929688
GPU Memory: 17.84863 MB
Epoch 5000/100000, Loss: 350.3096923828125
GPU Memory: 17.84863 MB
Epoch 6000/100000, Loss: 243.48789978027344
GPU Memory: 17.84863 MB
Epoch 7000/100000, Loss: 215.83001708984375
GPU Memory: 17.84863 MB
Epoch 8000/100000, Loss: 201.13385009765625
GPU Memory: 17.84863 MB
Epoch 9000/100000, Loss: 187.6595458984375
GPU Memory: 17.84863 MB
Epoch 10000/100000, Loss: 174.8900604248047
GPU Memory: 17.84863 MB
Epoch 11000/100000, Loss: 162.7533416748047
GPU Memory: 17.84863 MB
Epoch 12000/100000, Loss: 151.43360900878906
GPU Memory: 17.84863 MB
Epoch 13000/100000, Loss: 140.67462158203125
GPU Memory: 17.84863 MB
Epoch 14000/100000, Loss: 129.92051696777344
GPU Memory: 17.84863 MB
Epoch 15000/100000, Loss: 118.28768920898438
GPU Me

### Testing

In [32]:
torch.__version__

'2.9.1+cu128'

In [33]:
!uv add torchmetrics

Resolved 142 packages in 19ms
Audited 48 packages in 7ms


In [34]:
# set to model.eval()
model.eval()

with torch.no_grad():
    y_pred = model(X_test_tensor)
        
loss = criterion(y_pred, y_test_tensor)
print(loss)

from torchmetrics.functional.regression import r2_score

r2 = r2_score(y_pred, y_test_tensor)
print(f'R-squared on the test data: {r2.item()}')

with torch.no_grad():
    y_pred_train = model(X_train_tensor)
        

training_loss = criterion(y_pred_train, y_train_tensor)
print(training_loss)

r2 = r2_score(y_pred_train, y_train_tensor)
print(f'R-squared on the train data: {r2.item()}')

tensor(49.05998611, device='cuda:0')
R-squared on the test data: 0.8096064925193787
tensor(1.47564852, device='cuda:0')
R-squared on the train data: 0.994805634021759


## Batch Training

In [35]:
from torch.utils.data import TensorDataset, DataLoader

In [36]:
# Create TensorDataset and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Set batch size
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [37]:
# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNN(in_features=X_train_tensor.shape[1]).to(device)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001) 

# Training with batching
epochs = 100000
train_losses = []
val_losses = []

for epoch in range(epochs):
    # Training phase
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
    
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
        if torch.cuda.is_available():
            print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB')

# Evaluate on full test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor.to(device))
    test_loss = criterion(test_outputs, y_test_tensor.to(device))
    print(f'\nFinal Test Loss: {test_loss.item():.6f}')

Epoch 1000/100000, Train Loss: 142.387759, Val Loss: 135.610432
GPU Memory: 17.94 MB
Epoch 2000/100000, Train Loss: 59.163587, Val Loss: 67.816739
GPU Memory: 17.94 MB
Epoch 3000/100000, Train Loss: 37.657586, Val Loss: 46.071567
GPU Memory: 17.94 MB
Epoch 4000/100000, Train Loss: 29.067130, Val Loss: 38.537034
GPU Memory: 17.94 MB
Epoch 5000/100000, Train Loss: 23.447879, Val Loss: 35.089897
GPU Memory: 17.94 MB
Epoch 6000/100000, Train Loss: 19.811931, Val Loss: 33.351218
GPU Memory: 17.94 MB
Epoch 7000/100000, Train Loss: 17.495304, Val Loss: 32.532169
GPU Memory: 17.94 MB
Epoch 8000/100000, Train Loss: 15.548795, Val Loss: 31.615659
GPU Memory: 17.94 MB
Epoch 9000/100000, Train Loss: 13.925197, Val Loss: 31.021209
GPU Memory: 17.94 MB
Epoch 10000/100000, Train Loss: 12.682908, Val Loss: 30.302996
GPU Memory: 17.94 MB
Epoch 11000/100000, Train Loss: 11.581237, Val Loss: 29.655443
GPU Memory: 17.94 MB


KeyboardInterrupt: 

Evaluation

In [38]:
# set to model.eval()
model.eval()

with torch.no_grad():
    y_pred = model(X_test_tensor)
        
loss = criterion(y_pred, y_test_tensor)
print(loss)

from torchmetrics.functional.regression import r2_score

r2 = r2_score(y_pred, y_test_tensor)
print(f'R-squared on the test data: {r2.item()}')

with torch.no_grad():
    y_pred_train = model(X_train_tensor)
        

training_loss = criterion(y_pred_train, y_train_tensor)
print(training_loss)

r2 = r2_score(y_pred_train, y_train_tensor)
print(f'R-squared on the train data: {r2.item()}')

tensor(30.45409393, device='cuda:0')
R-squared on the test data: 0.8818128108978271
tensor(10.72957802, device='cuda:0')
R-squared on the train data: 0.9622312784194946


## Batch Normalization

In [39]:
class SimpleNNWithBN(nn.Module):
    def __init__(self, in_features):
        super(SimpleNNWithBN, self).__init__()
        self.in_features = in_features
        
        # Layers with Batch Normalization
        self.fc1 = nn.Linear(self.in_features, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 16)
        self.bn3 = nn.BatchNorm1d(16)
        self.fc4 = nn.Linear(16, 1)  # Output layer - no BN
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2) 

    def forward(self, x):
        # Layer 1: Linear -> BatchNorm -> Activation
        x = self.fc1(x)
        x = self.bn1(x)  # BatchNorm before activation
        x = self.relu(x)
        x = self.dropout(x) 
        
        # Layer 2
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        # Layer 3
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        # Output layer (no BN, no activation for regression)
        x = self.fc4(x)
        return x

In [40]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Set batch size
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [41]:
model = SimpleNNWithBN(in_features=X_train_tensor.shape[1]).to(device)

# Training parameters
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # Added weight decay
epochs = 20000


In [42]:
rain_losses = []
val_losses = []

for epoch in range(epochs):
    # Training phase
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
    
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
        if torch.cuda.is_available():
            print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB')

# Evaluate on full test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor.to(device))
    test_loss = criterion(test_outputs, y_test_tensor.to(device))
    print(f'\nFinal Test Loss: {test_loss.item():.6f}')

Epoch 1000/20000, Train Loss: 61.450430, Val Loss: 30.609270
GPU Memory: 17.96 MB
Epoch 2000/20000, Train Loss: 51.772677, Val Loss: 30.311113
GPU Memory: 17.96 MB
Epoch 3000/20000, Train Loss: 45.048604, Val Loss: 31.859599
GPU Memory: 17.96 MB
Epoch 4000/20000, Train Loss: 43.330981, Val Loss: 36.244450
GPU Memory: 17.96 MB
Epoch 5000/20000, Train Loss: 34.684938, Val Loss: 34.913653
GPU Memory: 17.96 MB
Epoch 6000/20000, Train Loss: 47.106311, Val Loss: 44.148481
GPU Memory: 17.96 MB
Epoch 7000/20000, Train Loss: 40.334719, Val Loss: 47.030235
GPU Memory: 17.96 MB
Epoch 8000/20000, Train Loss: 36.134100, Val Loss: 40.972626
GPU Memory: 17.96 MB
Epoch 9000/20000, Train Loss: 39.805650, Val Loss: 39.121226
GPU Memory: 17.96 MB
Epoch 10000/20000, Train Loss: 36.919312, Val Loss: 36.223961
GPU Memory: 17.96 MB
Epoch 11000/20000, Train Loss: 38.216916, Val Loss: 40.845976
GPU Memory: 17.96 MB
Epoch 12000/20000, Train Loss: 36.999253, Val Loss: 41.902352
GPU Memory: 17.96 MB
Epoch 13000/2

## Early stopping and Rate Scheduler

In [43]:
class EarlyStopping:
    """Simplified early stopping based on validation loss"""
    def __init__(self, patience=10, min_delta=0.0, min_epochs=100):
        self.patience = patience
        self.min_delta = min_delta
        self.min_epochs = min_epochs
        self.counter = 0
        self.best_loss = None
        self.should_stop = False
        
    def __call__(self, epoch, val_loss):
        if epoch < self.min_epochs:
            return False
            
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"Early stopping triggered at epoch {epoch}")
                return True
        else:
            self.best_loss = val_loss
            self.counter = 0
        return False


In [51]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNNWithBN(in_features=X_train_tensor.shape[1]).to(device)

# Training parameters
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 20000

In [52]:
early_stopping = EarlyStopping(patience=50, min_epochs=200)

In [53]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=30)

In [54]:
warmup_epochs = 100
warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.001, end_factor=1.0, total_iters=warmup_epochs)

In [55]:
train_losses = []
val_losses = []

In [56]:
for epoch in range(epochs):
    
    
    
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    if epoch < warmup_epochs:
        warmup_scheduler.step()
        
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
    
    if epoch >= warmup_epochs:
        scheduler.step(avg_val_loss)
    

    if early_stopping(epoch, avg_val_loss):
        print(f"Training stopped at epoch {epoch+1}")
        break

    if (epoch + 1) % 1000 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.6f}, '
              f'Val Loss: {avg_val_loss:.6f}, LR: {current_lr:.2e}')
        if torch.cuda.is_available():
            print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB')


model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor.to(device))
    test_loss = criterion(test_outputs, y_test_tensor.to(device))
    print(f'\nFinal Test Loss: {test_loss.item():.6f}')

Early stopping triggered at epoch 253
Training stopped at epoch 254

Final Test Loss: 31.798716


In [57]:
# set to model.eval()
model.eval()

with torch.no_grad():
    y_pred = model(X_test_tensor)
        
loss = criterion(y_pred, y_test_tensor)
print(loss)

from torchmetrics.functional.regression import r2_score

r2 = r2_score(y_pred, y_test_tensor)
print(f'R-squared on the test data: {r2.item()}')

with torch.no_grad():
    y_pred_train = model(X_train_tensor)
        

training_loss = criterion(y_pred_train, y_train_tensor)
print(training_loss)

r2 = r2_score(y_pred_train, y_train_tensor)
print(f'R-squared on the train data: {r2.item()}')

tensor(31.79871559, device='cuda:0')
R-squared on the test data: 0.876594603061676
tensor(20.74698067, device='cuda:0')
R-squared on the train data: 0.9269694685935974


# Neural Network with Residual Connection

In [58]:
class ResidualBlock(nn.Module):
    def __init__(self, features):
        super(ResidualBlock, self).__init__()
        
        # Main path (F(x))
        self.linear1 = nn.Linear(features, features)
        self.bn1 = nn.BatchNorm1d(features)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(features, features)
        self.bn2 = nn.BatchNorm1d(features)
        
    def forward(self, x):
        # Save input for skip connection
        identity = x
        
        # Main path: F(x)
        out = self.linear1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.linear2(out)
        out = self.bn2(out)
        
        # Skip connection: add input to output
        out = out + identity 
        
        # Final activation
        out = self.relu(out)
        
        return out

In [59]:
class ResidualNetwork(nn.Module):
    """Deep network WITH skip connections (Residual Network)"""
    def __init__(self, input_dim, hidden_dim=128, num_blocks=5, output_dim=1):
        super(ResidualNetwork, self).__init__()
        
        # Input projection
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.input_bn = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        
        # Residual blocks
        self.res_blocks = nn.ModuleList([
            ResidualBlock(hidden_dim) for _ in range(num_blocks)
        ])
        
        # Output layer
        self.output_layer = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # Input
        x = self.input_layer(x)
        x = self.input_bn(x)
        x = self.relu(x)
        
        # Residual blocks with skip connections
        for block in self.res_blocks:
            x = block(x)  # Each block: y = F(x) + x
        
        # Output
        x = self.output_layer(x)
        return x

But we can try a little simpler stuff

In [72]:
class SimpleNNWithBNSkip(nn.Module):
    def __init__(self, in_features):
        super(SimpleNNWithBNSkip, self).__init__()
        self.in_features = in_features
        
        # Layers with Batch Normalization
        self.fc1 = nn.Linear(self.in_features, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 16)
        self.bn3 = nn.BatchNorm1d(16)
        self.fc4 = nn.Linear(16, 1)  # Output layer - no BN
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2) 

        # because input and output features are different
        # by default we have 
        out_features = 1
        self.projection = nn.Linear(in_features, out_features)

    def forward(self, x):
        identity = x  # save the input
        # Layer 1: Linear -> BatchNorm -> Activation
        x = self.fc1(x)
        x = self.bn1(x)  # BatchNorm before activation
        x = self.relu(x)
        x = self.dropout(x) 
        
        # Layer 2
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        # Layer 3
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.fc4(x)

        identity = self.projection(identity)
        return x + identity

In [73]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Set batch size
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [74]:
model = SimpleNNWithBNSkip(in_features=X_train_tensor.shape[1]).to(device)

# Training parameters
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # Added weight decay
epochs = 20000


In [75]:
rain_losses = []
val_losses = []

for epoch in range(epochs):
    # Training phase
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
    
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
        if torch.cuda.is_available():
            print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB')

# Evaluate on full test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor.to(device))
    test_loss = criterion(test_outputs, y_test_tensor.to(device))
    print(f'\nFinal Test Loss: {test_loss.item():.6f}')

Epoch 1000/20000, Train Loss: 37.152470, Val Loss: 36.094006
GPU Memory: 18.33 MB
Epoch 2000/20000, Train Loss: 27.440666, Val Loss: 32.066908
GPU Memory: 18.33 MB
Epoch 3000/20000, Train Loss: 25.979272, Val Loss: 36.223816
GPU Memory: 18.33 MB
Epoch 4000/20000, Train Loss: 23.413445, Val Loss: 35.201470
GPU Memory: 18.33 MB
Epoch 5000/20000, Train Loss: 26.224768, Val Loss: 36.603139
GPU Memory: 18.33 MB
Epoch 6000/20000, Train Loss: 24.796298, Val Loss: 39.840645
GPU Memory: 18.33 MB
Epoch 7000/20000, Train Loss: 23.498904, Val Loss: 39.557159
GPU Memory: 18.33 MB
Epoch 8000/20000, Train Loss: 22.532016, Val Loss: 38.787832
GPU Memory: 18.33 MB
Epoch 9000/20000, Train Loss: 25.087429, Val Loss: 35.652602
GPU Memory: 18.33 MB
Epoch 10000/20000, Train Loss: 22.272954, Val Loss: 35.635161
GPU Memory: 18.33 MB
Epoch 11000/20000, Train Loss: 22.466980, Val Loss: 36.450426
GPU Memory: 18.33 MB
Epoch 12000/20000, Train Loss: 23.082333, Val Loss: 42.813428
GPU Memory: 18.33 MB
Epoch 13000/2

In [76]:
# set to model.eval()
model.eval()

with torch.no_grad():
    y_pred = model(X_test_tensor)
        
loss = criterion(y_pred, y_test_tensor)
print(loss)

from torchmetrics.functional.regression import r2_score

r2 = r2_score(y_pred, y_test_tensor)
print(f'R-squared on the test data: {r2.item()}')

with torch.no_grad():
    y_pred_train = model(X_train_tensor)
        

training_loss = criterion(y_pred_train, y_train_tensor)
print(training_loss)

r2 = r2_score(y_pred_train, y_train_tensor)
print(f'R-squared on the train data: {r2.item()}')

tensor(36.36101151, device='cuda:0')
R-squared on the test data: 0.858889102935791
tensor(27.86545181, device='cuda:0')
R-squared on the train data: 0.9019120335578918
